[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/garrygu/newegg-ai-workshop/blob/main/lv1-beginner-v2/instructor_guides/Session_3_Model_Builder.ipynb)


# 🛠️ Session 3 — Model Builder (Teacher)

This notebook is **ONLY FOR THE INSTRUCTOR**. You run this notebook *before* class to train the brain (`classifier.pt`) that your students will use in their game!

This notebook generates a `ResNet18` model trained on your custom pictures, perfectly matching the setup in the new Student v2 notebook.

## Step 1 — Organize Your Folders

1. Create a folder named `game_assets/` right next to this notebook.
2. Inside `game_assets/`, create a folder for each label your game has (for example, a folder named `cat` and a folder named `dog`).
3. Drop some pictures into those folders! The more pictures, the smarter the AI.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## Step 2 — Load Your Pictures

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

try:
    dataset = datasets.ImageFolder('game_assets', transform=transform)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)
    classes = dataset.classes
    print(f'✅ Found {len(dataset)} images belonging to {len(classes)} classes: {classes}')
except Exception as e:
    print('⚠️ Error finding folders: Did you create the game_assets folder and put labeled sub-folders inside it?')

## Step 3 — Build the Brain

We start with a `ResNet18` that already knows a lot about the world (`weights='DEFAULT'`). We chop off its final classification layer, and replace it with a brand new one that perfectly matches the number of categories in your folder!

In [ ]:
model = models.resnet18(weights='DEFAULT')

# Freeze the old brain so it trains super fast
for param in model.parameters():
    param.requires_grad = False

# Plug in our custom voicebox
if 'classes' in locals():
    model.fc = nn.Linear(model.fc.in_features, len(classes))
    model = model.to(device)
    print('🤖 Brain ready for training!')
else:
    print('⚠️ Cannot build brain: classes not loaded yet.')

## Step 4 — Train!

In [ ]:
if 'classes' in locals():
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.fc.parameters(), lr=0.005)

    EPOCHS = 10
    print('🏋️ Training started...')

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        if len(dataloader) > 0:
            print(f'Epoch {epoch+1}/{EPOCHS} complete. Loss: {total_loss/len(dataloader):.4f}')
    
    print('\n✅ Training done!')
else:
    print('⚠️ Cannot train. Run Step 2 first to load pictures.')

## Step 5 — Save the Save File (`.pt`)

This generates the file you will share with your students!

In [ ]:
Path('models').mkdir(exist_ok=True)

if 'classes' in locals():
    torch.save(model.state_dict(), 'models/classifier.pt')
    print('💾 Successfully saved the AI brain to models/classifier.pt!')
    print('\nNext steps: Share this file with your students so they can load it!')
else:
    print('⚠️ Cannot save model. Run the steps above first.')
